# Week 8B: Web scraping practice and dynamic content

Practice scraping static pages, then use Selenium to interact with dynamic websites.

**Lecture guide:** Static-page exercises → Dynamic content → Property assessments → Court search

## Week 8B agenda: web scraping

**Last time:**
- Why web scraping? 
- Getting familiar with the Web
- Web scraping: extracting data from static sites

**Today:**
- Practice with web scraping
- How to deal with dynamic content

In [1]:
# Start with the usual imports
import pandas as pd
from bs4 import BeautifulSoup
import requests

request_headers = {"User-Agent": "Mozilla/5.0"}

## Part 1: Web scraping exercises

For each of the exercises, use the Web Inspector to inspect the structure of the relevant web page, and identify the HTML content you will need to scrape with Python.


### Exercise 1: How many millions of people are currently experiencing drought?

Relevant URL: [https://www.drought.gov/current-conditions](https://www.drought.gov/current-conditions)

**Hint:** We're interested in just a single HTML element so you can inspect the website, identify the right element, and copy the selector for the element.

In [2]:
url = "https://www.drought.gov/current-conditions"
response = requests.get(url, headers=request_headers, timeout=30)
response.raise_for_status()
soup2 = BeautifulSoup(response.content, "html.parser")

In [3]:
# Inspect the statistic's accompanying text, not just its position.
# The old positional selector now points to the number of states, not people.
selector = ".field--name-field-number-stat"
population_stats = [
    tag for tag in soup2.select(selector)
    if "people in the U.S." in tag.parent.get_text(" ", strip=True)
]
if len(population_stats) != 1:
    raise ValueError("Population statistic not identified. Inspect the live drought page.")

In [4]:
population_stats[0].get_text(" ", strip=True)

Out[4]: '101.9 Million'


### Exercise 2: Scrape the Weitzman School directory

The Weitzman School lists its directory on [the people page](https://www.design.upenn.edu/people/list). Extract each person's name, title, and associated department.

Like the City health centers demo in Lecture 9A, select the repeated containers and loop over their fields. Here each person is wrapped in a `<div class="views-row">`, with `.title`, `.meta`, and `.dept` elements inside it.

Before using `rows[0]`, check the HTTP response and confirm that the selector matched rows. An empty list may mean an error page was downloaded, the HTML changed, or the content needs JavaScript. It does not mean that there are no people in the directory.

In [5]:
url = "https://www.design.upenn.edu/people/list"
response = requests.get(url, headers=request_headers, timeout=30)
response.raise_for_status()
soup3 = BeautifulSoup(response.content, "html.parser")

In [6]:
rows = soup3.select(".views-row")
if not rows:
    raise ValueError(
        f"No directory rows found at {response.url} (HTTP {response.status_code}). "
        "Inspect response.text and compare the live page's HTML with the selector."
    )

In [7]:
len(rows)

Out[7]: 595


In [8]:
# Cell above checks that rows is nonempty before indexing.
print(rows[0].prettify())

<div class="views-row">
 <a class="list-item profile-item" href="/people/huda-abdelnur">
  <span class="text">
   <span class="title heading-5">
    Huda Abdelnur
   </span>
   <span class="meta body-small">
    MCP Student
    <i>
     //
    </i>
    Admissions Ambassador
   </span>
  </span>
  <span class="dept body-subhead">
   Admissions
  </span>
  <span class="arrow">
   <span aria-hidden="true" class="fa fa-arrow-right">
   </span>
  </span>
 </a>
 <div class="views-field views-field-edit-node edit">
 </div>
</div>



In [9]:
records = []
for row in rows:
    fields = {
        "person": row.select_one(".title"),
        "title": row.select_one(".meta"),
        "dept": row.select_one(".dept"),
    }
    if fields["person"] is None:
        raise ValueError("A directory row has no name. Inspect the page structure.")
    records.append({
        key: tag.get_text(" ", strip=True) if tag is not None else None
        for key, tag in fields.items()
    })

data = pd.DataFrame(records)
data.head(10)

Out[9]: 
                  person  ...                                        dept
0          Huda Abdelnur  ...                                  Admissions
1               Tom Abel  ...  Center for Environmental Building & Design
2           Tanisha Gerg  ...              Penn Animation as Research Lab
3     Dr. Mostafa Akbari  ...                                Architecture
4     Masoud  Akbarzadeh  ...                                Architecture
5             Scott Aker  ...                                Architecture
6      Mohamad Al Khayer  ...                                Architecture
7        Farah Al Qasimi  ...                                   Fine Arts
8    Dr. Asseel Al-Ragam  ...                                Architecture
9  Dr. Sarah Saad Alajmi  ...                                Architecture

[10 rows x 3 columns]


## Part 2: What about dynamic content?

How do you scrape data that is loaded via Javascript or only appears after user interaction?

<center>
    <img src="imgs/yoda.png" width=700>
</center>

> **Note:**
>
> **Note: web browser needed**
> 
> You'll need a web browser installed to use `selenium`, e.g., FireFox, Google Chrome, Edge, etc.

### Selenium 

- Designed as a framework for testing webpages during development
- Provides an interface to interact with webpages just as a user would 
- Becoming increasingly popular for web scraping dynamic content from pages

### Two common use cases

Two cases when `requests.get()` won't work:

1. Many sites load content via javascript, so requests.get() won't return anything. It doesn't load any javascript on the page, just returns the static HTML content.
1. If the site requires user interaction via buttons, dropdowns, etc., then get requests won't be able to show that information.


**But how to solve them?**

1. Look on the "Network" tab for the external requests that the javascript is using to pull the data. 
1. Selenium to the rescue! 

Let's try it out!

### Example #1: Philly's Property App

- URL: [https://property.phila.gov/](https://property.phila.gov/)
- Let's see if we can extract the property assessment values for my old apartment in Fairmount

**Problem:** We'll need to enter an address, click a button, and THEN scrape the information.

#### 1. Initialize the web driver

The initialization steps will depend on which browser you want to use!

In [ ]:
# !pip install selenium

In [ ]:
# Import the webdriver from selenium
from selenium import webdriver

> **Important:**
>
> **Important: Working on Binder**
> 
> If you are working on Binder, you'll need to use FireFox in "headless" mode, which prevents a browser window from opening.
> 
> If you are working locally, it's better to run with the default options — you'll be able to see the browser window open and change as we perform the web scraping.

##### Using Google Chrome

In [ ]:
# UNCOMMENT BELOW TO USE CHROME

driver = webdriver.Chrome()

##### Using Firefox

If you are working on Binder, use the below code!

In [ ]:
# UNCOMMENT BELOW IF ON BINDER

# options = webdriver.FirefoxOptions()

# IF ON BINDER, RUN IN "HEADLESS" MODE (NO BROWSER WINDOW IS OPENED)
# COMMENT THIS LINE IF WORKING LOCALLY
# options.add_argument("--headless")

# Initialize
# driver = webdriver.Firefox(options=options)

##### Using Microsoft Edge

In [ ]:
# UNCOMMENT BELOW TO USE MICROSOFT EDGE

#driver = webdriver.Edge()

#### 2. Navigate to the URL

In [ ]:
url = "https://property.phila.gov/"
driver.get(url)

#### 3. Select the address input element

After it loads, let's take a look in the web inspector. We'll need to identify the search bar so we can input our desired address.

It looks like the search bar has a class of `"pvm-search-control-input"`.

In [ ]:
# Use the dot syntax for class selectors
address_input_selector = ".pvm-search-control-input"

We can select it directly in selenium using the `driver.find_element()` function and telling selenium we are using a css selector to do the selection.

In [ ]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

wait = WebDriverWait(driver, 30)

In [ ]:
address_input = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, address_input_selector))
)

#### 4. Type the address

Use the `send_keys()` function to enter the text into the input element:

In [ ]:
address_input.clear()
address_input.send_keys("1739 Wallace St #102")

#### 5. Select the search button and click it

Using the web inspector, it looks like both the clear and search buttons have the class of `"pvm-search-control-button"`. Let's select both, choose the second button, and click it.

In [ ]:
button_selector = ".pvm-search-control-button"
buttons = driver.find_elements(By.CSS_SELECTOR, button_selector)

In [ ]:
if len(buttons) != 2:
    raise ValueError("Expected clear and search buttons. Inspect the live search controls.")
delete_button, search_button = buttons

In [ ]:
# Click it
search_button.click()

#### 6. Open the property details before waiting for assessments

**Checkpoint:** look at the Selenium browser after the search click. A search and a property selection are separate interactions.

- If matching properties appear, click the entry for **1739 Wallace St, unit 102** in the browser to open its details. Confirm the address before continuing.
- If you are still on the search form or see an error, stop here. Read the error, confirm the address was entered, and check whether clicking Search actually submitted it. Waiting for a table cannot fix a failed search.
- Once the property details are open, look for **Valuation History**. It is loaded asynchronously.

The app's source attaches a property-detail action to result-row clicks. This classroom demo uses a manual selection checkpoint so we do not accidentally open a different property.

The app also reuses `id="ownerProperties"` for different tables. The first one may show **Owner / OPA Account Number**, not assessment values. Selecting `tables[0]` or `tables[1]` is unreliable.

In [ ]:
assessment_columns = [
    "Year", "Market Value", "Taxable Land", "Taxable Improvement",
    "Exempt Land", "Exempt Improvement",
]

#### Diagnose what the browser actually loaded

Run the diagnostic below before the wait. It prints the current URL and each table's headings and row count. This is more useful than increasing the timeout.

- **No tables:** the browser may still be on the search form, an error page, or a loading screen.
- **Street Address / Owner / OPA Account Number:** these describe search results or ownership, not the assessment history. Open the matching property's details.
- **Year / Market Value / Taxable Land …:** the assessment table is present. Check its row count and field structure.

If the browser displays an error, include its text when reporting the problem. A timeout alone does not identify the cause.

In [ ]:
page = BeautifulSoup(driver.page_source, "html.parser")
print("Current URL:", driver.current_url)
for i, candidate in enumerate(page.select("table")):
    headings = [tag.get_text(" ", strip=True) for tag in candidate.select("th")]
    print(i, headings, "rows:", len(candidate.select("tbody > tr")))

In [ ]:
def find_assessment_table(browser):
    page = BeautifulSoup(browser.page_source, "html.parser")
    for candidate in page.select("table"):
        headings = [" ".join(tag.get_text(" ", strip=True).split()) for tag in candidate.select("thead th")]
        if headings != assessment_columns:
            continue
        data_rows = candidate.select("tbody > tr")
        if data_rows and all(
            len(row.select("td")) == len(assessment_columns)
            for row in data_rows
        ):
            return candidate
    return False

try:
    table = wait.until(find_assessment_table)
except TimeoutException:
    page = BeautifulSoup(driver.page_source, "html.parser")
    print("Current URL:", driver.current_url)
    for i, candidate in enumerate(page.select("table")):
        headings = [tag.get_text(" ", strip=True) for tag in candidate.select("th")]
        print(i, headings, "rows:", len(candidate.select("tbody > tr")))
    raise TimeoutException(
        "No populated assessment table was found. If the browser is still on "
        "the search page or shows an error, fix the search first. If it shows "
        "results, open the matching property. Compare the printed headings "
        "with assessment_columns before rerunning this cell."
    ) from None

In [ ]:
[tag.get_text(" ", strip=True) for tag in table.select("thead th")]

**Why the earlier `IndexError` happened:**

`rows[0]` asks for the first item in a list. If the selected table has an empty `<tbody>`, `table.select("tbody tr")` returns `[]`, which has no first item. We must select the correct table and wait for its rows, then check the list before indexing it.

Preview the assessment table selected by its headings:

In [ ]:
print("\n".join(table.prettify().splitlines()[:30]))

In [ ]:
# Keep the table returned by the wait; do not replace it with tables[0].
type(table)

Select all the rows in the table (the `<tr>` elements):

In [ ]:
rows = table.select("tbody > tr")
if not rows:
    raise ValueError("The selected assessment table has no rows. Rerun the wait cell.")
len(rows)

Now, we can take a look at each `<tr>` element and extract each cell's data in the `<td>` cell

In [ ]:
print(rows[0].prettify())

Read the text from each `<td>` cell. Some values may be wrapped in `<b>` tags, but selecting `td` works with or without that extra formatting.

In [ ]:
row = rows[0]
row

In [ ]:
scraped_data = []
for row in rows:
    row_data = [cell.get_text(" ", strip=True) for cell in row.select("td")]
    if len(row_data) != len(assessment_columns):
        raise ValueError("Assessment row does not match the expected six columns.")
    scraped_data.append(row_data)
scraped_data[:3]

Make it into a DataFrame:

In [ ]:
assessments = pd.DataFrame(scraped_data, columns=assessment_columns)
assessments

The assessment DataFrame contains the rows from the live Valuation History table. Values remain strings for now; remove currency formatting before numerical analysis.

### Example #2: Scraping the Philadelphia Municipal Courts portal

- URL: [https://ujsportal.pacourts.us/CaseSearch](https://ujsportal.pacourts.us/CaseSearch)
- Given a Police incident number, we'll see if there is an associated court case with the incident

**Problem:** we'll need to click several buttons before we can see the info we want!

#### Strategy for the court search

Strategy:

- Rely on the Web Inspector to identify specific elements of the webpage
- Use Selenium to interact with the webpage
    - Change dropdown elements
    - Click buttons

#### 1. Open the URL

In [ ]:
# Open the URL
url = "https://ujsportal.pacourts.us/CaseSearch"
driver.get(url)

#### 2. Create a dropdown "Select" element

We'll need to: 
- Select the dropdown element on the main page by its ID
- Initialize a `selenium` `Select()` object

In [ ]:
# Use the Web Inspector to get the css selector of the dropdown select element
dropdown_selector = "#SearchBy-Control > select" 

In [ ]:
dropdown = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, dropdown_selector))
)

Create the Select object:

In [ ]:
from selenium.webdriver.support.ui import Select

# Initialize a Select object
dropdown_select = Select(dropdown)

#### 3. Change the selected text in the dropdown

Change the selected element: "Police Incident/Complaint Number" 

In [ ]:
# Set the selected text in the dropdown element
dropdown_select.select_by_visible_text("Incident Number")

#### 4. Set the incident number 

In [ ]:
incident_input_selector = "#IncidentNumber-Control > input[type=text]"
incident_input = wait.until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, incident_input_selector))
)

In [ ]:
# Clear any existing entry
incident_input.clear()

# Input our example incident number
incident_input.send_keys("1725088232")

#### 5. Click the search button!

In [ ]:
# Submit the search
search_button_selector = "#btnSearch"
search_button = driver.find_element(By.CSS_SELECTOR, search_button_selector)
search_button.click()

#### 6. Use BeautifulSoup to parse the results

- Use the `page_source` attribute to get the current HTML displayed on the page
- Initialize a "soup" object with the HTML

In [ ]:
# Wait for the result grid after submitting the search.
# If the portal displays a no-results message instead, inspect that message
# in the browser; do not assume an empty/missing table is a case record.
wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "#caseSearchResultGrid tbody")))
courtsSoup = BeautifulSoup(driver.page_source, "html.parser")

Now we can:
- Identify the element holding all of the results
- Within this container, find the `<table>` element and each `<tr>` element within the table

In [ ]:
# Select the results container by its ID 
results_table = courtsSoup.select_one("#caseSearchResultGrid")

In [ ]:
if results_table is None:
    raise ValueError("Court results table not found. Inspect the browser's search response.")
results_rows = results_table.select("tbody > tr")
# Inspect any no-results message before interpreting rows as cases.

**Example:** The number of court cases

In [ ]:
# Number of court cases
number_of_cases = len(results_rows)
print(f"Number of courts cases: {number_of_cases}")

**Example:** Extract the text elements from the first row of the results

In [ ]:
first_row = results_rows[0] if results_rows else None
if first_row is None:
    print("No case rows found. Check the portal's result message and search criteria.")

In [ ]:
if first_row is not None:
    print(first_row.prettify())

In [ ]:
if first_row is not None:
    for cell in first_row.select("td"):
        text = cell.get_text(" ", strip=True)
        if text:
            print(text)

#### 7. Close the driver!

In [ ]:
driver.quit()

## That's it!

- Next week: Part 2 of "getting data" with APIs
- See you on Monday!